[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/19_cadenas_de_markov/notebooks/aplicaciones/03_letras_y_lenguaje.ipynb)

# Letras y Lenguaje: Cadenas de Markov para Texto

**Módulo 19 — Cadenas de Markov · Notebook 03 (Aplicación)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

COLORS = {
    "red": "#E94F37", "orange": "#F28C28", "blue": "#2E86AB",
    "green": "#27AE60", "purple": "#8E44AD", "gray": "#7F8C8D",
}

np.random.seed(42)
print("Listo ✓")

---
## Parte 1: Replicando el análisis de Markov (1913)

En 1913, Andrei Markov analizó la novela en verso *Eugenio Oneguin* de Alexander Pushkin.
Clasificó **20,000 caracteres** consecutivos como **vocal (V)** o **consonante (C)** y contó las
transiciones entre ellos. Con eso demostró que las letras *no* son independientes: la
probabilidad de que la siguiente letra sea vocal depende de si la actual es vocal o consonante.

Fue la primera aplicación empírica de lo que hoy llamamos *cadena de Markov*.

Nosotros haremos exactamente lo mismo, pero con texto en **español**: un pasaje de
*Don Quijote de la Mancha* de Miguel de Cervantes.

In [ ]:
# Pasaje de Don Quijote de la Mancha (Capítulo I, adaptado)
TEXTO_QUIJOTE = """
En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho
tiempo que vivia un hidalgo de los de lanza en astillero, adarga antigua,
rocin flaco y galgo corredor. Una olla de algo mas vaca que carnero,
salpicon las mas noches, duelos y quebrantos los sabados, lantejas los
viernes, algun palomino de anyadidura los domingos, consumian las tres
partes de su hacienda. El resto della concluian sayo de velarte, calzas
de velludo para las fiestas con sus pantuflos de lo mesmo, y los dias de
entre semana se honraba con su vellori de lo mas fino. Tenia en su casa
una ama que pasaba de los cuarenta, y una sobrina que no llegaba a los
veinte, y un mozo de campo y plaza, que asi ensillaba el rocin como tomaba
la podadera. Frisaba la edad de nuestro hidalgo con los cincuenta anyos,
era de complexion recia, seco de carnes, enjuto de rostro, gran madrugador
y amigo de la caza. Quieren decir que tenia el sobrenombre de Quijada o
Quesada, que en esto hay alguna diferencia en los autores que deste caso
escriben, aunque por conjeturas verosimiles se deja entender que se
llamaba Quijana. Pero esto importa poco a nuestro cuento, basta que en
la narracion del no se salga un punto de la verdad. Es pues de saber que
este sobredicho hidalgo, los ratos que estaba ocioso, que eran los mas
del anyo, se daba a leer libros de caballerias con tanta aficion y gusto,
que olvido casi de todo punto el ejercicio de la caza, y aun la
administracion de su hacienda. Y llego a tanto su curiosidad y desatino
en esto, que vendio muchas fanegas de tierra de sembradura para comprar
libros de caballerias en que leer, y asi llevo a su casa todos cuantos
pudo haber dellos. Y de todos ningunos le parecian tan bien como los que
compuso el famoso Feliciano de Silva, porque la claridad de su prosa y
aquellas entricadas razones suyas le parecian de perlas, y mas cuando
llegaba a leer aquellos requiebros y cartas de desafios, donde en muchas
partes hallaba escrito la razon de la sinrazon que a mi razon se hace,
de tal manera mi razon enflaquece, que con razon me quejo de la vuestra
fermosura. Y tambien cuando leia los altos cielos que de vuestra
divinidad divinamente con las estrellas os fortifican y os hacen
merecedora del merecimiento que merece la vuestra grandeza.
"""

def limpiar_texto(texto):
    """Limpia texto: minúsculas, sin acentos, solo a-z y espacio."""
    texto = texto.lower()
    # Reemplazar caracteres acentuados
    reemplazos = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
        'ü': 'u', 'ñ': 'n',
    }
    for viejo, nuevo in reemplazos.items():
        texto = texto.replace(viejo, nuevo)
    # Mantener solo a-z y espacio
    texto = ''.join(c if c.isalpha() or c == ' ' else ' ' for c in texto)
    # Colapsar espacios múltiples
    while '  ' in texto:
        texto = texto.replace('  ', ' ')
    return texto.strip()

def clasificar_vc(texto):
    """Clasifica cada carácter como V (vocal) o C (consonante). Ignora espacios."""
    vocales = set('aeiou')
    resultado = []
    for c in texto:
        if c == ' ':
            continue
        elif c in vocales:
            resultado.append('V')
        else:
            resultado.append('C')
    return resultado

texto_limpio = limpiar_texto(TEXTO_QUIJOTE)
secuencia_vc = clasificar_vc(texto_limpio)

print(f"Longitud del texto limpio: {len(texto_limpio)} caracteres")
print(f"Caracteres clasificados (sin espacios): {len(secuencia_vc)}")
print(f"Primeros 80 caracteres: {texto_limpio[:80]}")
vc_str = ''.join(secuencia_vc[:60])
print(f"Clasificación V/C:      {vc_str}")

In [ ]:
# Contar transiciones V->V, V->C, C->V, C->C
conteos = {'VV': 0, 'VC': 0, 'CV': 0, 'CC': 0}
for i in range(len(secuencia_vc) - 1):
    par = secuencia_vc[i] + secuencia_vc[i+1]
    conteos[par] += 1

print("Conteos de transiciones:")
for par, n in conteos.items():
    print(f"  {par[0]} -> {par[1]}: {n}")

# Matriz de transición 2x2 (filas: estado actual, columnas: estado siguiente)
# Orden: [V, C]
P_vc = np.zeros((2, 2))
P_vc[0, 0] = conteos['VV']  # V->V
P_vc[0, 1] = conteos['VC']  # V->C
P_vc[1, 0] = conteos['CV']  # C->V
P_vc[1, 1] = conteos['CC']  # C->C

# Normalizar filas
P_vc = P_vc / P_vc.sum(axis=1, keepdims=True)

print("\nMatriz de transición P (filas: estado actual, columnas: siguiente):")
print(f"         V        C")
print(f"  V  [{P_vc[0,0]:.4f}   {P_vc[0,1]:.4f}]")
print(f"  C  [{P_vc[1,0]:.4f}   {P_vc[1,1]:.4f}]")

**Derivación algebraica.** Buscamos $\pi$ tal que $\pi \mathbf{P} = \pi$:

$$\pi_V \cdot P_{VV} + \pi_C \cdot P_{CV} = \pi_V$$
$$\pi_V + \pi_C = 1$$

Sustituyendo $\pi_C = 1 - \pi_V$ en la primera ecuación y simplificando:

$$\pi_V = \frac{P_{CV}}{P_{VC} + P_{CV}}$$

donde $P_{VC} = P(V \to C)$ y $P_{CV} = P(C \to V)$ son las probabilidades de transición estimadas del texto.

In [ ]:
# Distribución estacionaria: pi P = pi, con pi_V + pi_C = 1
# De pi P = pi:
#   pi_V = pi_V * P(V->V) + pi_C * P(C->V)
#   pi_V (1 - P_vc[0,0] + P_vc[1,0]) = P_vc[1,0]
#   pi_V = P_vc[1,0] / (1 - P_vc[0,0] + P_vc[1,0])

pi_V = P_vc[1, 0] / (1 - P_vc[0, 0] + P_vc[1, 0])
pi_C = 1 - pi_V

print(f"Distribución estacionaria (español, Quijote):")
print(f"  pi_V = {pi_V:.4f}")
print(f"  pi_C = {pi_C:.4f}")

# Verificación: frecuencia empírica
freq_V = sum(1 for c in secuencia_vc if c == 'V') / len(secuencia_vc)
print(f"\nFrecuencia empírica de vocales: {freq_V:.4f}")
print(f"(Debe ser cercana a pi_V = {pi_V:.4f})")

### Comparación entre idiomas

Los valores de Markov (1913) para el ruso de *Eugenio Oneguin* se pueden comparar con
nuestros resultados para el español de *Don Quijote*:

| | Español (Quijote) | Ruso (Eugenio Oneguin, 1913) |
|---|---|---|
| P(V&#8594;V) | *[calculado arriba]* | 0.128 |
| P(V&#8594;C) | *[calculado arriba]* | 0.872 |
| P(C&#8594;V) | *[calculado arriba]* | 0.663 |
| P(C&#8594;C) | *[calculado arriba]* | 0.337 |
| &#960;_V | *[calculado arriba]* | 0.432 |

**Observación:** A pesar de ser idiomas muy diferentes (español: romance; ruso: eslavo),
la estructura es similar: **vocales y consonantes tienden a alternarse**.
La transición más probable desde una vocal es hacia una consonante, y viceversa.
Esta alternancia es una propiedad casi universal de los idiomas humanos hablados,
porque facilita la pronunciación.

La propiedad de Markov captura este patrón lingüístico fundamental.

> **Fuente:** Markov, A.A. (1913). *Ejemplo de investigación estadística del texto de "Eugenio Oneguin" ilustrando la asociación de pruebas en cadena.* Publicado en las Actas de la Academia Imperial de Ciencias de San Petersburgo.

In [ ]:
# Simular la cadena V/C y mostrar convergencia a pi_V
n_steps = 5000
states = np.zeros(n_steps, dtype=int)  # 0=V, 1=C
states[0] = 0  # empezamos en V

for t in range(1, n_steps):
    if np.random.rand() < P_vc[states[t-1], 0]:
        states[t] = 0  # V
    else:
        states[t] = 1  # C

# Frecuencia acumulada de vocales
cumulative_V = np.cumsum(states == 0) / np.arange(1, n_steps + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cumulative_V, color=COLORS["blue"], lw=1.2, label="Frecuencia acumulada de V")
ax.axhline(pi_V, color=COLORS["red"], ls="--", lw=2, label=f"$\\pi_V = {pi_V:.4f}$")
ax.set_xlabel("Paso $t$")
ax.set_ylabel("Frecuencia de vocales")
ax.set_title("Convergencia de la cadena V/C a la distribución estacionaria")
ax.legend()
ax.set_ylim(0.3, 0.7)
plt.tight_layout()
plt.show()

---
## Parte 2: Matriz de transición completa (27 estados)

Pasamos de la clasificación binaria V/C a un modelo de **27 estados**:
las 26 letras del alfabeto (a-z) más el espacio.

Cada entrada $P_{ij}$ de la matriz será la probabilidad de que, dado que el carácter
actual es $i$, el siguiente sea $j$.

In [ ]:
# Definir los 27 estados: espacio + a-z
LETRAS = ' abcdefghijklmnopqrstuvwxyz'
n_estados = len(LETRAS)  # 27
letra_a_idx = {c: i for i, c in enumerate(LETRAS)}

# Contar transiciones
conteos_27 = np.zeros((n_estados, n_estados))
for i in range(len(texto_limpio) - 1):
    c_actual = texto_limpio[i]
    c_siguiente = texto_limpio[i + 1]
    if c_actual in letra_a_idx and c_siguiente in letra_a_idx:
        conteos_27[letra_a_idx[c_actual], letra_a_idx[c_siguiente]] += 1

# Normalizar filas (evitar división por cero)
sumas_filas = conteos_27.sum(axis=1, keepdims=True)
sumas_filas[sumas_filas == 0] = 1  # evitar NaN
P_27 = conteos_27 / sumas_filas

print(f"Matriz de transición: {P_27.shape}")
print(f"Suma de cada fila (debe ser 1.0): {P_27.sum(axis=1)[:5]} ...")
print(f"\nEjemplo: después de 'q', las probabilidades son:")
idx_q = letra_a_idx['q']
for j in range(n_estados):
    if P_27[idx_q, j] > 0.01:
        print(f"  q -> '{LETRAS[j]}': {P_27[idx_q, j]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Reemplazar ceros con NaN para mejor visualización
P_display = P_27.copy()
P_display[P_display == 0] = np.nan

im = ax.imshow(P_display, cmap='YlOrRd', aspect='equal', interpolation='nearest')

# Etiquetar ejes
etiquetas = ['SP'] + list('abcdefghijklmnopqrstuvwxyz')
ax.set_xticks(range(n_estados))
ax.set_yticks(range(n_estados))
ax.set_xticklabels(etiquetas, fontsize=9)
ax.set_yticklabels(etiquetas, fontsize=9)
ax.set_xlabel("Siguiente carácter", fontsize=12)
ax.set_ylabel("Carácter actual", fontsize=12)
ax.set_title("Matriz de transición: letras del español", fontsize=14)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Probabilidad de transición", fontsize=11)

plt.tight_layout()
plt.show()

### Patrones en la matriz

Observa la estructura en el heatmap:

- **Después de 'q'**, casi siempre viene 'u' (fila 'q' tiene un solo punto brillante).
- **Después del espacio**, las letras iniciales más comunes son 'd', 'e', 'l', 'c', 's', 'a'
  (las palabras más frecuentes del español: *de, el, la, con, se, a, ...).*
- **Después de vocales**, la probabilidad se distribuye entre muchas consonantes.
- **Después de consonantes**, hay alta probabilidad de vocales (la alternancia V/C que vimos en la Parte 1).
- Muchas celdas son cero o casi cero: combinaciones como 'qx', 'zz', 'wk' no existen en español.

**La estructura del idioma español está codificada en esta matriz.**

In [ ]:
def generar_texto(P, letras, n_chars, inicio=None, semilla=42):
    """Genera texto a partir de una cadena de Markov sobre caracteres.

    Parámetros
    ----------
    P : ndarray (k, k)
        Matriz de transición.
    letras : str
        String con los k caracteres (en el mismo orden que P).
    n_chars : int
        Número de caracteres a generar.
    inicio : str or None
        Carácter inicial. Si None, se elige al azar.
    semilla : int
        Semilla para reproducibilidad.

    Retorna
    -------
    str : texto generado.
    """
    rng = np.random.default_rng(semilla)
    k = len(letras)
    idx_map = {c: i for i, c in enumerate(letras)}

    if inicio is None:
        estado = rng.integers(0, k)
    else:
        estado = idx_map[inicio]

    resultado = [letras[estado]]
    for _ in range(n_chars - 1):
        estado = rng.choice(k, p=P[estado])
        resultado.append(letras[estado])

    return ''.join(resultado)

In [ ]:
n_gen = 200

# --- Orden 0: uniforme sobre los 27 caracteres ---
rng0 = np.random.default_rng(42)
texto_orden0 = ''.join(rng0.choice(list(LETRAS), size=n_gen))

# --- Orden 1: muestreo según frecuencias marginales (distribución estacionaria) ---
# Calcular la distribución estacionaria de la matriz 27x27
# Método: vector propio izquierdo con eigenvalor 1
eigenvalues, eigenvectors = np.linalg.eig(P_27.T)
# Encontrar el eigenvalor más cercano a 1
idx_1 = np.argmin(np.abs(eigenvalues - 1.0))
pi_27 = np.real(eigenvectors[:, idx_1])
pi_27 = pi_27 / pi_27.sum()  # normalizar
pi_27 = np.abs(pi_27)  # asegurar no-negatividad (errores numéricos)
pi_27 = pi_27 / pi_27.sum()  # re-normalizar

rng1 = np.random.default_rng(42)
texto_orden1 = ''.join(rng1.choice(list(LETRAS), size=n_gen, p=pi_27))

# --- Cadena de Markov: muestreo con la matriz de transición ---
texto_markov = generar_texto(P_27, LETRAS, n_gen, inicio=' ', semilla=42)

print("Orden 0 (uniforme):")
print(f'  "{texto_orden0}"')
print()
print("Orden 1 (frecuencias):")
print(f'  "{texto_orden1}"')
print()
print("Cadena de Markov (orden 1, transiciones):")
print(f'  "{texto_markov}"')

### Comparación de los tres niveles

1. **Orden 0 (uniforme):** Texto completamente aleatorio. Ninguna estructura reconocible.
   Las letras raras (q, x, z, w) aparecen con la misma frecuencia que las comunes.

2. **Orden 1 (frecuencias):** Las letras comunes del español (e, a, o, s, n) dominan,
   pero las combinaciones no respetan la estructura del idioma. Se ve algo más "español"
   solo porque las frecuencias son correctas.

3. **Cadena de Markov:** El texto es reconocible como *pseudo-español*. Aparecen
   combinaciones que suenan a palabras reales. No es perfecto, pero la mejora es
   dramática.

Agregar más contexto (orden 2: condicionar en los 2 caracteres previos; orden 3, etc.)
mejoraría progresivamente la calidad del texto generado.

---
## Parte 3: La conexión con Shannon

En 1948, Claude Shannon publicó *"A Mathematical Theory of Communication"*, el artículo
fundacional de la teoría de la información. Shannon usó **exactamente esta jerarquía** de
modelos de caracteres (orden 0, orden 1, cadena de Markov) para definir la **entropía**
y la **redundancia** del idioma inglés.

Shannon demostró que:

- El inglés tiene aproximadamente **1 bit de entropía por carácter** (en el límite de contextos largos).
- Una selección uniforme de 27 caracteres tiene $\log_2(27) \approx 4.75$ bits por carácter.
- Una cadena de Markov de orden 1 reduce esto a aproximadamente **3 bits**.
- Contextos más largos reducen la entropía aún más.

### De Markov a los modelos de lenguaje modernos

Los modelos de lenguaje modernos (GPT, Claude) son la extensión extrema de la idea de Markov:
en lugar de condicionar en 1-2 caracteres previos, condicionan en **miles de tokens previos**.
La base matemática es la misma idea que Markov tuvo en 1913: las dependencias secuenciales
crean estructura, y esa estructura se puede capturar con probabilidades condicionales.

| Modelo | Contexto | Entropía aprox. (bits/char) |
|--------|----------|----------------------------|
| Uniforme (27 chars) | 0 | ~4.75 |
| Frecuencias (orden 1) | 0 | ~4.0 |
| Markov (orden 1) | 1 char | ~3.0 |
| Markov (orden 2-3) | 2-3 chars | ~2.5 |
| Modelo de lenguaje moderno | miles de tokens | ~1.0 |

In [ ]:
# Entropía de cada modelo

# 1. Uniforme: H = log2(27)
H_uniforme = np.log2(n_estados)

# 2. Basado en frecuencias (distribución estacionaria): H = -sum(pi * log2(pi))
# Evitar log(0)
pi_safe = pi_27.copy()
pi_safe[pi_safe == 0] = 1e-15
H_frecuencias = -np.sum(pi_27 * np.log2(pi_safe))

# 3. Cadena de Markov: H = -sum_i pi_i sum_j P_ij * log2(P_ij)
H_markov = 0.0
for i in range(n_estados):
    for j in range(n_estados):
        if P_27[i, j] > 0:
            H_markov -= pi_27[i] * P_27[i, j] * np.log2(P_27[i, j])

print("Comparación de entropías (bits por carácter):")
print("=" * 50)
print(f"  Uniforme (27 caracteres):     {H_uniforme:.4f} bits")
print(f"  Frecuencias (estacionaria):   {H_frecuencias:.4f} bits")
print(f"  Cadena de Markov (orden 1):   {H_markov:.4f} bits")
print()
print(f"Reducción uniforme -> frecuencias:  {H_uniforme - H_frecuencias:.4f} bits ({(1 - H_frecuencias/H_uniforme)*100:.1f}%)")
print(f"Reducción uniforme -> Markov:       {H_uniforme - H_markov:.4f} bits ({(1 - H_markov/H_uniforme)*100:.1f}%)")
print()
print("Cada reducción en entropía refleja estructura del idioma")
print("que el modelo logra capturar.")

### Comparación con el inglés (Shannon, 1948)

Shannon calculó la entropía del inglés a distintos órdenes en su artículo fundacional:

| Nivel | Español (Quijote) | Inglés (Shannon 1948) |
|---|---|---|
| Orden 0 (uniforme) | ~4.76 bits | ~4.76 bits (log₂ 27) |
| Orden 1 (frecuencias) | ~4.0 bits | ~4.03 bits |
| Orden 2 (Markov) | ~2.9 bits | ~2.8 bits |

Los valores son notablemente similares a pesar de las diferencias entre idiomas. La estructura estadística del lenguaje — la tendencia a alternar vocales y consonantes, las frecuencias desiguales de letras — es un fenómeno universal que Markov descubrió en 1913 y Shannon formalizó en 1948.

---
## Resumen

1. **El análisis de Markov (1913)** fue la primera demostración empírica de que las letras
   en un texto no son independientes. Replicamos su análisis con español y obtuvimos
   resultados estructuralmente similares a los del ruso.

2. **La matriz de transición 27x27** codifica la estructura del español a nivel de caracteres.
   El heatmap revela patrones como la alternancia vocal-consonante, las combinaciones
   obligatorias (q-u) y las letras iniciales de palabra más frecuentes.

3. **La generación de texto** muestra cómo cada nivel de sofisticación (uniforme, frecuencias,
   cadena de Markov) produce texto progresivamente más parecido al español real.

4. **La entropía de Shannon** cuantifica exactamente cuánta estructura captura cada modelo.
   La cadena de Markov reduce significativamente la entropía respecto al modelo uniforme,
   pero aún queda mucha estructura por capturar (contextos más largos, gramática, semántica).

5. **De Markov a GPT:** La línea directa va de Markov (1913) a Shannon (1948) a n-gramas
   a redes neuronales a transformers. La idea fundamental es siempre la misma:
   **modelar la distribución condicional del siguiente símbolo dado el contexto**.